In [1]:
# run_load_data.ipynb
# Run this in JupyterLab to load data to SQL Server

import pandas as pd
import pyodbc
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("LOADING DATA TO SQL SERVER")
print("="*60)

# ============================================
# 1. CONNECTION SETUP
# ============================================
print("\n[1] Setting up connection...")

# Connection string - UPDATE THIS!
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=LAPTOP-8G4F56T7\\SQLEXPRESS;"  
    "DATABASE=AirbnbLisbon;"
    "Trusted_Connection=yes;"  # Windows Authentication
)

try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    print(" Connected to SQL Server successfully!")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\n Troubleshooting tips:")
    print("1. Make sure SQL Server is running")
    print("2. Check server name (try: localhost\\SQLEXPRESS)")
    print("3. Verify database 'AirbnbLisbon' exists")
    exit()



LOADING DATA TO SQL SERVER

[1] Setting up connection...
 Connected to SQL Server successfully!


In [2]:
# ============================================
# 2. LOAD DATA
# ============================================
import numpy as np
print("\n[2] Loading enriched data...")

project_root = Path.cwd().parent
processed_path = project_root / "data" / "processed" / "listings_enriched.csv"

if not processed_path.exists():
    print(f" File not found: {processed_path}")
    print("Make sure you've run the data cleaning notebook first!")
    exit()

df = pd.read_csv(processed_path)
print(f" Loaded {len(df):,} rows from {processed_path}")

# Clean data - replace NaN with None for SQL
df = df.replace({np.nan: None})


[2] Loading enriched data...
 Loaded 24,876 rows from C:\Users\User\Documents\GitHub\expernetic-data-engineer-assessment\data\processed\listings_enriched.csv


In [3]:
# ============================================
# 3. CLEAR TABLES (IF THEY EXIST)
# ============================================
print("\n[3] Clearing existing data...")

try:
    cursor.execute("DELETE FROM fact_performance")
    conn.commit()
    print("  - Cleared fact_performance")
except:
    print("  - fact_performance already empty")

try:
    cursor.execute("DELETE FROM dim_listing")
    conn.commit()
    print("  - Cleared dim_listing")
except:
    print("  - dim_listing already empty")

try:
    cursor.execute("DELETE FROM dim_host")
    conn.commit()
    print("  - Cleared dim_host")
except:
    print("  - dim_host already empty")

try:
    cursor.execute("DELETE FROM dim_neighbourhood")
    conn.commit()
    print("  - Cleared dim_neighbourhood")
except:
    print("  - dim_neighbourhood already empty")

# ============================================
# 4. LOAD DIM_NEIGHBOURHOOD
# ============================================
print("\n[4] Loading neighbourhoods...")

neighbourhoods = df[['neighbourhood', 'neighbourhood_group']].drop_duplicates()
for _, row in neighbourhoods.iterrows():
    try:
        # Check if exists
        cursor.execute("""
            SELECT COUNT(*) FROM dim_neighbourhood 
            WHERE neighbourhood = ? AND neighbourhood_group = ?
        """, row['neighbourhood'], row['neighbourhood_group'])
        
        if cursor.fetchone()[0] == 0:
            cursor.execute("""
                INSERT INTO dim_neighbourhood (neighbourhood_group, neighbourhood)
                VALUES (?, ?)
            """, row['neighbourhood_group'], row['neighbourhood'])
    except Exception as e:
        print(f"  - Error: {e}")
        continue

conn.commit()
print(f" Loaded {len(neighbourhoods)} neighbourhoods")


[3] Clearing existing data...
  - Cleared fact_performance
  - Cleared dim_listing
  - Cleared dim_host
  - Cleared dim_neighbourhood

[4] Loading neighbourhoods...
 Loaded 130 neighbourhoods


In [4]:
# ============================================
# 5. LOAD DIM_HOST
# ============================================
print("\n[5] Loading hosts...")

hosts = df[['host_id', 'host_name', 'host_tenure_years', 'is_superhost', 'host_total_listings']].drop_duplicates('host_id')
hosts_loaded = 0

for _, row in hosts.iterrows():
    try:
        host_id = int(row['host_id']) if pd.notna(row['host_id']) else None
        if host_id is None:
            continue
            
        # Check if exists
        cursor.execute("SELECT COUNT(*) FROM dim_host WHERE host_id = ?", host_id)
        if cursor.fetchone()[0] == 0:
            cursor.execute("""
                INSERT INTO dim_host (host_id, host_name, host_tenure_years, is_superhost, host_total_listings)
                VALUES (?, ?, ?, ?, ?)
            """, 
            host_id,
            str(row['host_name'])[:200] if pd.notna(row['host_name']) else None,
            float(row['host_tenure_years']) if pd.notna(row['host_tenure_years']) else 0,
            str(row['is_superhost']) if pd.notna(row['is_superhost']) else 'f',
            int(row['host_total_listings']) if pd.notna(row['host_total_listings']) else 0
            )
            hosts_loaded += 1
            
        if hosts_loaded % 1000 == 0:
            conn.commit()
            print(f"  - Loaded {hosts_loaded} hosts...")
    except Exception as e:
        continue

conn.commit()
print(f" Loaded {hosts_loaded} hosts")


[5] Loading hosts...
  - Loaded 1000 hosts...
  - Loaded 2000 hosts...
  - Loaded 3000 hosts...
  - Loaded 4000 hosts...
  - Loaded 5000 hosts...
  - Loaded 6000 hosts...
  - Loaded 7000 hosts...
  - Loaded 8000 hosts...
  - Loaded 9000 hosts...
 Loaded 9521 hosts


In [5]:
# ============================================
# 6. LOAD DIM_LISTING
# ============================================
print("\n[6] Loading listings...")

listings_loaded = 0

for _, row in df.iterrows():
    try:
        listing_id = int(row['listing_id']) if pd.notna(row['listing_id']) else None
        if listing_id is None:
            continue
            
        # Check if exists
        cursor.execute("SELECT COUNT(*) FROM dim_listing WHERE listing_id = ?", listing_id)
        if cursor.fetchone()[0] == 0:
            # Safely convert values
            accommodates = int(row['accommodates']) if pd.notna(row['accommodates']) else 0
            bedrooms = float(row['bedrooms']) if pd.notna(row['bedrooms']) else 0
            bathrooms = float(row['bathrooms']) if pd.notna(row['bathrooms']) else 0
            min_nights = int(row['min_nights']) if pd.notna(row['min_nights']) else 1
            max_nights = int(row['max_nights']) if pd.notna(row['max_nights']) else 365
            
            # Cap max_nights to prevent overflow
            if max_nights > 999999:
                max_nights = 999999
            
            cursor.execute("""
                INSERT INTO dim_listing (
                    listing_id, listing_name, property_type, room_type, 
                    accommodates, bedrooms, bathrooms, min_nights, max_nights,
                    latitude, longitude, neighbourhood, neighbourhood_group
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            listing_id,
            str(row['listing_name'])[:500] if pd.notna(row['listing_name']) else None,
            str(row['property_type']) if pd.notna(row['property_type']) else None,
            str(row['room_type']) if pd.notna(row['room_type']) else None,
            accommodates,
            bedrooms,
            bathrooms,
            min_nights,
            max_nights,
            float(row['latitude']) if pd.notna(row['latitude']) else 0,
            float(row['longitude']) if pd.notna(row['longitude']) else 0,
            str(row['neighbourhood']) if pd.notna(row['neighbourhood']) else None,
            str(row['neighbourhood_group']) if pd.notna(row['neighbourhood_group']) else None
            )
            listings_loaded += 1
            
        if listings_loaded % 1000 == 0:
            conn.commit()
            print(f"  - Loaded {listings_loaded} listings...")
    except Exception as e:
        print(f"  - Skipped listing {listing_id}: {e}")
        continue

conn.commit()
print(f" Loaded {listings_loaded} listings")


[6] Loading listings...
  - Loaded 1000 listings...
  - Loaded 2000 listings...
  - Loaded 3000 listings...
  - Loaded 4000 listings...
  - Loaded 5000 listings...
  - Loaded 6000 listings...
  - Loaded 7000 listings...
  - Loaded 8000 listings...
  - Loaded 9000 listings...
  - Loaded 10000 listings...
  - Loaded 11000 listings...
  - Loaded 12000 listings...
  - Loaded 13000 listings...
  - Loaded 14000 listings...
  - Loaded 15000 listings...
  - Loaded 16000 listings...
  - Loaded 17000 listings...
  - Loaded 18000 listings...
  - Loaded 19000 listings...
  - Loaded 20000 listings...
  - Loaded 21000 listings...
  - Loaded 22000 listings...
  - Loaded 23000 listings...
  - Loaded 24000 listings...
 Loaded 24876 listings


In [6]:
# ============================================
# 6. LOAD FACT_PERFORMANCE
# ============================================
print("\n[6] Loading fact_performance...")

try:
    cursor.execute("TRUNCATE TABLE fact_performance")
    conn.commit()
except:
    conn.commit()

count = 0
for _, row in df.iterrows():
    try:
        # Get keys
        cursor.execute("SELECT listing_key FROM dim_listing WHERE listing_id = ?", int(row['listing_id']))
        listing_key_result = cursor.fetchone()
        if not listing_key_result:
            continue
        listing_key = listing_key_result[0]
        
        cursor.execute("SELECT host_key FROM dim_host WHERE host_id = ?", int(row['host_id']))
        host_key_result = cursor.fetchone()
        if not host_key_result:
            continue
        host_key = host_key_result[0]
        
        cursor.execute("""
            SELECT neighbourhood_key FROM dim_neighbourhood 
            WHERE neighbourhood = ? AND neighbourhood_group = ?
        """, 
        str(row['neighbourhood']) if pd.notna(row['neighbourhood']) else '',
        str(row['neighbourhood_group']) if pd.notna(row['neighbourhood_group']) else ''
        )
        neighbourhood_key_result = cursor.fetchone()
        if not neighbourhood_key_result:
            continue
        neighbourhood_key = neighbourhood_key_result[0]
        
        # Insert fact
        cursor.execute("""
            INSERT INTO fact_performance (
                listing_key, host_key, neighbourhood_key, 
                price, price_per_bedroom,
                availability_365, occupancy_rate, estimated_revenue,
                review_count, review_score, 
                first_review_date, last_review_date
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        listing_key,
        host_key,
        neighbourhood_key,
        float(row['price']) if pd.notna(row['price']) else None,
        float(row['price_per_bedroom']) if pd.notna(row['price_per_bedroom']) else None,
        int(row['availability_365']) if pd.notna(row['availability_365']) else None,
        float(row['occupancy_rate']) if pd.notna(row['occupancy_rate']) else None,
        float(row['estimated_revenue']) if pd.notna(row['estimated_revenue']) else None,
        int(row['review_count']) if pd.notna(row['review_count']) else 0,
        float(row['review_score']) if pd.notna(row['review_score']) else None,
        row['first_review_date'] if pd.notna(row['first_review_date']) else None,
        row['last_review_date'] if pd.notna(row['last_review_date']) else None
        )
        count += 1
        if count % 1000 == 0:
            print(f"  - Loaded {count:,} records...")
            conn.commit()
    except Exception as e:
        print(f"  - Error on row {count}: {e}")
        continue

conn.commit()
print(f" Loaded {count:,} fact records")


[6] Loading fact_performance...
  - Loaded 1,000 records...
  - Loaded 2,000 records...
  - Loaded 3,000 records...
  - Loaded 4,000 records...
  - Loaded 5,000 records...
  - Loaded 6,000 records...
  - Loaded 7,000 records...
  - Loaded 8,000 records...
  - Loaded 9,000 records...
  - Loaded 10,000 records...
  - Loaded 11,000 records...
  - Loaded 12,000 records...
  - Loaded 13,000 records...
  - Loaded 14,000 records...
  - Loaded 15,000 records...
  - Loaded 16,000 records...
  - Loaded 17,000 records...
  - Loaded 18,000 records...
  - Loaded 19,000 records...
  - Loaded 20,000 records...
  - Loaded 21,000 records...
  - Loaded 22,000 records...
  - Loaded 23,000 records...
  - Loaded 24,000 records...
 Loaded 24,876 fact records


In [7]:
# ============================================
# 8. VERIFICATION
# ============================================
print("\n[8] Verification...")

try:
    cursor.execute("SELECT COUNT(*) FROM dim_neighbourhood")
    print(f"   dim_neighbourhood: {cursor.fetchone()[0]:,} rows")
except:
    print("   dim_neighbourhood: error")

try:
    cursor.execute("SELECT COUNT(*) FROM dim_host")
    print(f"   dim_host: {cursor.fetchone()[0]:,} rows")
except:
    print("   dim_host: error")

try:
    cursor.execute("SELECT COUNT(*) FROM dim_listing")
    print(f"   dim_listing: {cursor.fetchone()[0]:,} rows")
except:
    print("   dim_listing: error")

try:
    cursor.execute("SELECT COUNT(*) FROM fact_performance")
    print(f"   fact_performance: {cursor.fetchone()[0]:,} rows")
except:
    print("   fact_performance: error")




[8] Verification...
   dim_neighbourhood: 130 rows
   dim_host: 9,521 rows
   dim_listing: 24,876 rows
   fact_performance: 24,876 rows


In [8]:
# ============================================
# 9. COMPLETE
# ============================================
print("\n" + "="*60)
print(" DATA LOAD COMPLETED SUCCESSFULLY!")
print("="*60)

conn.close()


 DATA LOAD COMPLETED SUCCESSFULLY!
